# Threading vs Multiprocessing vs asyncio - Which Concurrency Tool When?

Difficulty: Advanced | ~60 min | Requires Lab 5 (recommended)

The same two jobs - CPU work and I/O work - run four ways. The winner flips between them.

In [ ]:
!pip install tabulate==0.10.0

## Section 0 - The three models at a glance

**Concurrency** means handling several tasks without waiting for each one. Which tool to use comes down to one question: is your program **computing** or **waiting**? Section 7 of the guide unpacks every row below, and the cells after this one prove them.

| | Threads | Processes | asyncio |
|---|---|---|---|
| What runs | OS threads inside one Python program | separate Python programs | coroutine tasks on one event loop |
| Memory | one shared heap that all threads see | each worker gets a private copy | one shared heap |
| CPU parallelism (GIL on) | no | yes | no |
| I/O concurrency | good | works, but heavy to start | best |
| Cost to start a worker | about 50-100 microseconds | about 100-500 ms on Windows | about 1 microsecond per task |
| Passing work and data | free, same objects | copied in and out (pickled) | free, same objects |
| Shared mutable state | possible, needs a `Lock` | not possible by default | safe from interruption |
| Failure mode | shared data gets corrupted | `BrokenProcessPool` | one blocking call freezes everything |

## Section 1 - The two workloads

`blur` is CPU work (pure Python loops, never waits). `fetch` is I/O work (a 50 ms pretend-wait). Both live in `workloads.py`, and Section 5 shows exactly why.

In [2]:
import time
from workloads import blur, fetch, IMAGES

# timeit wraps any callable and returns (result, elapsed_seconds).
# Every benchmark below is then one line instead of three.
def timeit(fn, *args):
    """Run fn(*args), return (result, elapsed_seconds)."""
    start = time.perf_counter()      # monotonic, high-resolution wall clock
    result = fn(*args)               # the work being measured
    return result, time.perf_counter() - start

# Sanity check: blur one image to prove the workload actually works.
blur(IMAGES[0])
print(f"Task: blur {len(IMAGES)} images of {len(IMAGES[0])}x{len(IMAGES[0])}")

Task: blur 100 images of 100x100


## Section 2 - Anatomy 1: what a thread actually is

Four threads, one shared list. The point is the memory model, not the API - and the danger arrives in Section 4.

In [3]:
import os
import threading
from workloads import probe_worker

# One shared list is the whole message bus: every thread appends to the *same*
# object, so nothing is copied and nothing is pickled.
observations = []

def inspect(label):
    """Thread body: run one probe, then hand the result back through the list."""
    observations.append(probe_worker(label))

# args=(n,) is how a Thread passes arguments to its target function.
names = [f"inspector-{i}" for i in range(4)]
probes = [threading.Thread(target=inspect, args=(n,), name=n) for n in names]

for thread in probes:
    thread.start()   # returns at once; the thread runs in the background
for thread in probes:
    thread.join()    # block until it finishes - without join, the work is not done

# Read the results back. Every pid is the same: one interpreter, one GIL.
pids = sorted({row["pid"] for row in observations})
thread_ids = {row["thread_id"] for row in observations}

print(f"parent pid       : {os.getpid()}")
print(f"pids seen        : {pids}   <- one interpreter, so one pid")
print(f"distinct threads : {len(thread_ids)}   <- four separate threads though")
print(f"thread names     : {sorted({row['thread_name'] for row in observations})}")

parent pid       : 21352
pids seen        : [21352]   <- one interpreter, so one pid
distinct threads : 4   <- four separate threads though
thread names     : ['inspector-0', 'inspector-1', 'inspector-2', 'inspector-3']


## Section 3 - Anatomy 2: the same probe, two pools

One function, two pool classes. The thread pool reports one pid; the process pool reports one pid per worker.

In [4]:
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

# 16 tasks x 50 ms each is more work than one worker can chew through, so every
# worker of each pool gets used. Otherwise the fast one drains the queue and the
# pid count lies. Same callable, same arguments: only the pool class differs.
labels = [f"task-{i}" for i in range(16)]
probes_by_pool = {}

# One loop over both pool classes, so the class is the only variable.
for label, pool_class in (("threads", ThreadPoolExecutor),
                          ("processes", ProcessPoolExecutor)):
    with pool_class(max_workers=4) as pool:
        probes_by_pool[label] = list(pool.map(probe_worker, labels))

# Threads report a single pid. Processes report one pid per live worker.
for label, runs in probes_by_pool.items():
    pids = sorted({row["pid"] for row in runs})
    print(f"{label:10} -> {len(pids)} distinct pid(s): {pids}")

threads    -> 1 distinct pid(s): [21352]
processes  -> 4 distinct pid(s): [12072, 12540, 20936, 35460]


## Section 4 - Anatomy 3: shared state, races, and locks

Four threads increment one shared counter: unguarded loses updates, a `Lock` loses none, and processes share nothing at all.

In [5]:
from workloads import bump_locked, bump_unlocked, counter_total, reset_counter

# Each of 4 workers tries to add STEPS. With no updates lost the total is
# STEPS * WORKERS, which is what EXPECTED means.
STEPS, WORKERS = 2000, 4
EXPECTED = STEPS * WORKERS

# 1. Four threads, no lock. Thread A reads the counter, yields the GIL, and then
#    writes back a number that no longer includes thread B's update - so B's
#    work silently vanishes. Nothing in the output looks wrong.
reset_counter()
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    pool.map(bump_unlocked, [STEPS] * WORKERS)      # results not needed
unlocked_total = counter_total()                    # read the one shared object
print(f"4 threads, no lock: {unlocked_total}/{EXPECTED} survived, "
      f"{EXPECTED - unlocked_total} lost to the race")

# 2. The identical loop and the identical threads, but every update now happens
#    under COUNTER_LOCK. Only one thread may be inside the `with` block at a
#    time, so every increment survives and the count is exact on every run.
reset_counter()
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    pool.map(bump_locked, [STEPS] * WORKERS)
locked_total = counter_total()
print(f"4 threads, Lock   : {locked_total}/{EXPECTED} survived, "
      f"{EXPECTED - locked_total} lost")

# 3. The same function in processes. No lock is even needed, because there is
#    nothing to share: each worker counts into its own copy of COUNTER, so every
#    total is a whole multiple of STEPS and the parent's counter stays at 0.
#    Which worker takes which task is up to the pool, so one worker may run
#    more than one of the 4 tasks and report 4000 instead of 2000.
reset_counter()
with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    per_worker = sorted(pool.map(bump_locked, [STEPS] * WORKERS))
print(f"4 processes, Lock : per-worker totals {per_worker} "
      f"(each a multiple of {STEPS}), parent still sees {counter_total()}")

4 threads, no lock: 2092/8000 survived, 5908 lost to the race
4 threads, Lock   : 8000/8000 survived, 0 lost


4 processes, Lock : per-worker totals [2000, 2000, 2000, 2000] (each a multiple of 2000), parent still sees 0


## Section 5 - Anatomy 4: start methods and the pickling rule

Under `spawn` a worker re-imports your module by name, which is why the work lives in `workloads.py` and a `lambda` fails.

In [6]:
import multiprocessing as mp

# The start method decides how a worker process is created. Windows has only spawn.
print(f"start method here : {mp.get_start_method()}")
print(f"available here    : {mp.get_all_start_methods()}")

# Under spawn the function is pickled and shipped by name. A lambda has no
# importable name, so this fails - and we catch it instead of crashing the cell.
try:
    with ProcessPoolExecutor(max_workers=2) as pool:
        pool.submit(lambda n: n, 1).result()
except Exception as exc:
    print(f"lambda via pool   : {type(exc).__name__}: {exc}")

start method here : spawn
available here    : ['spawn']


lambda via pool   : PicklingError: Can't pickle <function <lambda> at 0x000001DAD147A0C0>: attribute lookup <lambda> on __main__ failed

## Section 6 - Part 1: the sequential baseline (CPU)

Blur all 100 images on a single thread. This is the number every other strategy has to beat.

In [7]:
# The baseline: 100 blurs, one at a time, on one thread.
# _ discards the result list; cpu_seq keeps only the elapsed time.
_, cpu_seq = timeit(lambda: [blur(img) for img in IMAGES])
print(f"sequential: 100 images in {cpu_seq:.2f}s")

sequential: 100 images in 2.41s


## Section 7 - Part 2: threads (CPU)

Same work, spread across threads. Expect no speedup - the GIL allows one computing thread at a time, so this row is noise around 1.0x.

In [8]:
from concurrent.futures import ThreadPoolExecutor

# run_threads spreads work_fn across the items using a thread pool.
# Handing work over is free, because every thread sees the same objects.
def run_threads(work_fn, items):
    """Distribute work_fn across items using a thread pool."""
    with ThreadPoolExecutor() as pool:
        return list(pool.map(work_fn, items))

# The same 100 blurs, now spread across threads. The GIL still allows only one
# thread to run Python code at a time, so expect this near the baseline.
_, cpu_threads = timeit(lambda: run_threads(blur, IMAGES))
print(f"threads: 100 images in {cpu_threads:.2f}s")

threads: 100 images in 1.35s


## Section 8 - Part 2 (cont.): asyncio (CPU)

The coroutine never awaits, so the event loop never gets the wheel. Expect no speedup.

In [9]:
import asyncio

# This coroutine wraps the same comprehension but never awaits. `await` is the
# only place a coroutine hands control back, so the loop never gets the wheel.
async def process_images():
    return [blur(img) for img in IMAGES]

start = time.perf_counter()
await process_images()          # Jupyter already runs a loop, so await directly
cpu_async = time.perf_counter() - start
print(f"asyncio: 100 images in {cpu_async:.2f}s")

asyncio: 100 images in 1.93s


## Section 9 - Part 3: processes (CPU)

Separate interpreters, separate GILs, real parallelism - minus worker startup and pickling, which is why the row reads 1.1x-2.0x rather than 4x.

In [10]:
from concurrent.futures import ProcessPoolExecutor

# run_processes has the same shape as run_threads with a different pool class.
# Only module-level functions survive pickling, which is why `blur` is imported
# from workloads.py instead of being defined in this cell.
def run_processes(work_fn, items):
    """Distribute work_fn across items using a process pool."""
    with ProcessPoolExecutor() as pool:
        return list(pool.map(work_fn, items))

# Each worker process has its own GIL, so these blurs really do run in parallel.
_, cpu_proc = timeit(lambda: run_processes(blur, IMAGES))
print(f"processes: 100 images in {cpu_proc:.2f}s")

processes: 100 images in 1.01s


## Section 10 - Part 4: the same four strategies on I/O-bound work

The same four strategies on 100 fake 50 ms requests. Waiting is free to overlap, so the winner changes.

In [11]:
N = 100

# Sequential I/O: 100 x 50ms = ~5s, with the CPU idle the whole time.
_, io_seq_elapsed = timeit(lambda: [fetch(i) for i in range(N)])
print(f"sequential: {N} requests in {io_seq_elapsed:.2f}s")

# Threads on I/O: sleep() releases the GIL, so the waits overlap.
_, io_threads_elapsed = timeit(lambda: run_threads(fetch, range(N)))
print(f"threads: {N} requests in {io_threads_elapsed:.2f}s")

# asyncio on I/O: an `async def` that awaits lets the loop switch between all
# 100 tasks instead of waiting for each one in turn.
async def fetch_async(n):
    await asyncio.sleep(0.05)  # same 50 ms, but the loop switches tasks
    return n * 2

start = time.perf_counter()
await asyncio.gather(*[fetch_async(i) for i in range(N)])   # gather runs them together
io_async_elapsed = time.perf_counter() - start
print(f"asyncio: {N} requests in {io_async_elapsed:.2f}s")

# Processes on I/O: spawn overhead, and no GIL benefit to pay it back.
_, io_proc_elapsed = timeit(lambda: run_processes(fetch, range(N)))
print(f"processes: {N} requests in {io_proc_elapsed:.2f}s")

sequential: 100 requests in 5.06s


threads: 100 requests in 0.66s
asyncio: 100 requests in 0.06s


processes: 100 requests in 1.78s


## Section 11 - Reading the two ledgers

CPU work: processes win. I/O work: asyncio wins. The tools never changed - the job did.

In [12]:
from tabulate import tabulate

# print_ledger renders one comparison table: each strategy against the baseline.
def print_ledger(title, seq_time, times_dict):
    """Print a comparison table: each strategy vs the sequential baseline."""
    print(title)
    rows = [["sequential", f"{seq_time:.2f}s", "1x"]]
    for name, t in times_dict.items():
        speedup = seq_time / max(t, 1e-9)  # avoid division by zero
        rows.append([name, f"{t:.2f}s", f"{speedup:.2f}x"])
    print(tabulate(rows, headers=["strategy", "elapsed", "vs baseline"], tablefmt="grid"))

# CPU-bound ledger: processes should win, while threads and asyncio sit near baseline.
print_ledger("CPU-bound: blurring 100 images of 100x100 (lower is better)", cpu_seq,
             {"threads": cpu_threads, "asyncio": cpu_async, "processes": cpu_proc})

print()

# I/O-bound ledger: asyncio should crush everything, processes should come last.
print_ledger("I/O-bound: 100 fake 50 ms requests (lower is better)", io_seq_elapsed,
             {"threads": io_threads_elapsed, "asyncio": io_async_elapsed,
              "processes": io_proc_elapsed})

CPU-bound: blurring 100 images of 100x100 (lower is better)
+------------+-----------+---------------+
| strategy   | elapsed   | vs baseline   |
+============+===========+===============+
| sequential | 2.41s     | 1x            |
+------------+-----------+---------------+
| threads    | 1.35s     | 1.79x         |
+------------+-----------+---------------+
| asyncio    | 1.93s     | 1.25x         |
+------------+-----------+---------------+
| processes  | 1.01s     | 2.39x         |
+------------+-----------+---------------+

I/O-bound: 100 fake 50 ms requests (lower is better)
+------------+-----------+---------------+
| strategy   | elapsed   | vs baseline   |
+============+===========+===============+
| sequential | 5.06s     | 1x            |
+------------+-----------+---------------+
| threads    | 0.66s     | 7.63x         |
+------------+-----------+---------------+
| asyncio    | 0.06s     | 81.15x        |
+------------+-----------+---------------+
| processes  | 1.78s     |